In [1]:
# Instalar mrmr-selection se não estiver instalado
# !pip install mrmr-selection

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
from scipy.stats import pearsonr
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, auc
from sklearn.svm import LinearSVC, SVC
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LassoCV, Lasso, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
from sklearn.metrics import log_loss as sk_log_loss, matthews_corrcoef
from sklearn.ensemble import VotingClassifier

# Seleção de features mRMR
try:
    from mrmr import mrmr_classif
    MRMR_AVAILABLE = True
except ImportError:
    MRMR_AVAILABLE = False
    print("⚠️ mRMR não está instalado. Instale com: pip install mrmr-selection")

# Sklearn
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import VarianceThreshold

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

print("✅ Bibliotecas carregadas com sucesso!")

✅ Bibliotecas carregadas com sucesso!


In [2]:
# Carregar o dataset DoS
dos_path = Path("../data/processed/DoS_with_gap_features.csv")
df = pd.read_csv(dos_path)

print(f"📊 Shape do dataset: {df.shape}")
print(f"📝 Colunas: {df.shape[1]}")
print(f"📋 Registros: {df.shape[0]}")
print(f"\n🏷️ Distribuição do Label ('type'):")
print(df['type'].value_counts())

📊 Shape do dataset: (94625, 67)
📝 Colunas: 67
📋 Registros: 94625

🏷️ Distribuição do Label ('type'):
type
normal    49111
DoS       45514
Name: count, dtype: int64


In [10]:
df.head()

,frame.time_delta,frame.time_delta_displayed,frame.time_epoch,frame.time_invalid,frame.time_relative,ip.src,ip.dst,tcp.srcport,tcp.dstport,eth.src,eth.dst,frame.cap_len,frame.coloring_rule.name,frame.coloring_rule.string,frame.comment,frame.comment.expert,frame.encap_type,frame.file_off,frame.ignored,frame.incomplete,frame.interface_id,frame.interface_name,frame.len,frame.link_nr,frame.marked,frame.md5_hash,frame.number,frame.offset_shift,mqtt.clientid,mqtt.clientid_len,mqtt.conack.flags,mqtt.conack.flags.reserved,mqtt.conack.flags.sp,mqtt.conack.val,mqtt.conflag.cleansess,mqtt.conflag.passwd,mqtt.conflag.qos,mqtt.conflag.reserved,mqtt.conflag.retain,mqtt.conflag.uname,mqtt.conflag.willflag,mqtt.conflags,mqtt.dupflag,mqtt.hdrflags,mqtt.kalive,mqtt.len,mqtt.msg,mqtt.msgid,mqtt.msgtype,mqtt.passwd,mqtt.passwd_len,mqtt.protoname,mqtt.qos,mqtt.retain,mqtt.sub.qos,mqtt.suback.qos,mqtt.topic,mqtt.topic_len,mqtt.username,mqtt.username_len,mqtt.willmsg,mqtt.willmsg_len,mqtt.willtopic,mqtt.willtopic_len,type,publish_gap,connect_gap
0,0.000048,0.000048,1.522236e+09,NaN,449.592637,216.58.214.162,192.168.1.227,NaN,NaN,18:a6:f7:eb:77:26,48:5a:3f:93:39:9c,60,NaN,NaN,NaN,NaN,1,NaN,0,NaN,NaN,NaN,60,NaN,0,NaN,33957,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,normal,NaN,NaN
1,0.000063,0.000063,1.522236e+09,NaN,495.851280,192.168.1.196,192.168.1.171,52251.0,1883.0,30:5a:3a:62:72:80,74:d4:35:ef:e5:5a,1514,NaN,NaN,NaN,NaN,1,NaN,0,NaN,NaN,NaN,1514,NaN,0,NaN,39740,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0x00000030,NaN,170.0,C0a6CF1209Bc3Cba2bd29b1c16D78DAbB0CE8EF4Eaf0D2...,NaN,3.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,mqtt-malaria/beem.loadr-malaria-VirtualBox-706...,64.0,NaN,NaN,NaN,NaN,NaN,NaN,DoS,0.000000,NaN
2,0.000037,0.000037,1.522236e+09,NaN,502.861074,192.168.1.196,192.168.1.171,52192.0,1883.0,30:5a:3a:62:72:80,74:d4:35:ef:e5:5a,54,NaN,NaN,NaN,NaN,1,NaN,0,NaN,NaN,NaN,54,NaN,0,NaN,40517,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DoS,NaN,NaN
3,0.000145,0.000145,1.522236e+09,NaN,746.271645,192.168.1.196,192.168.1.171,52449.0,1883.0,30:5a:3a:62:72:80,74:d4:35:ef:e5:5a,1514,NaN,NaN,NaN,NaN,1,NaN,0,NaN,NaN,NaN,1514,NaN,0,NaN,51349,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0x00000030,NaN,174.0,86DEF0DCa0E921fd6A30Cafe345B932f71a4876Ed9C6EC...,NaN,3.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,mqtt-malaria/beem.loadr-malaria-VirtualBox-747...,63.0,NaN,NaN,NaN,NaN,NaN,NaN,DoS,250.420365,NaN
4,0.000060,0.000060,1.522236e+09,NaN,229.529526,192.168.1.196,192.168.1.171,51982.0,1883.0,30:5a:3a:62:72:80,74:d4:35:ef:e5:5a,219,NaN,NaN,NaN,NaN,1,NaN,0,NaN,NaN,NaN,219,NaN,0,NaN,16309,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0x00000030,NaN,168.0,cB3ff0eFA43d77aA4d8A6CBcD560b4D508Ee625B9Df73c...,NaN,3.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,mqtt-malaria/beem.loadr-malaria-VirtualBox-663...,62.0,NaN,NaN,NaN,NaN,NaN,NaN,DoS,-516.742119,NaN


In [3]:
# Selecionar apenas features numéricas
num_df = df.select_dtypes(include=['int64', 'float64'])

# Colunas a remover (irrelevantes para detecção de ataques)
colunas_para_remover = [
    'frame.time_delta_displayed', 'frame.time_epoch', 'frame.time_invalid', 
    'frame.time_relative', 'tcp.srcport', 'tcp.dstport', 
    'frame.coloring_rule.name', 'frame.coloring_rule.string', 
    'frame.comment', 'frame.comment.expert', 'frame.encap_type', 
    'frame.file_off', 'frame.ignored', 'frame.incomplete', 
    'frame.interface_id', 'frame.interface_name', 'frame.link_nr', 
    'frame.marked', 'frame.md5_hash', 'frame.number', 'frame.offset_shift',
    'mqtt.msgid', 'mqtt.username', 'mqtt.passwd', 
    'mqtt.willmsg', 'mqtt.willtopic'
]

# Remover colunas existentes no dataframe
colunas_existentes = [c for c in colunas_para_remover if c in num_df.columns]
X = num_df.drop(columns=colunas_existentes, errors='ignore')

# Tratar valores ausentes e infinitos
X = X.fillna(0)
X = X.replace([np.inf, -np.inf], 0)

# Encode do label (target)
le = LabelEncoder()
y = le.fit_transform(df['type'])

print(f"✅ Features numéricas após limpeza: {X.shape[1]}")
print(f"🎯 Classes: {le.classes_}")
print(f"📊 Shape X: {X.shape}, y: {y.shape}")

✅ Features numéricas após limpeza: 29
🎯 Classes: ['DoS' 'normal']
📊 Shape X: (94625, 29), y: (94625,)


In [9]:
X.head()

,frame.time_delta,frame.cap_len,frame.len,mqtt.clientid_len,mqtt.conack.flags.reserved,mqtt.conack.flags.sp,mqtt.conack.val,mqtt.conflag.cleansess,mqtt.conflag.passwd,mqtt.conflag.qos,mqtt.conflag.reserved,mqtt.conflag.retain,mqtt.conflag.uname,mqtt.conflag.willflag,mqtt.dupflag,mqtt.kalive,mqtt.len,mqtt.msgtype,mqtt.passwd_len,mqtt.qos,mqtt.retain,mqtt.sub.qos,mqtt.suback.qos,mqtt.topic_len,mqtt.username_len,mqtt.willmsg_len,mqtt.willtopic_len,publish_gap,connect_gap
0,0.000048,60,60,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
1,0.000063,1514,1514,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,170.0,3.0,0.0,0.0,0.0,0.0,0.0,64.0,0.0,0.0,0.0,0.000000,0.0
2,0.000037,54,54,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
3,0.000145,1514,1514,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,174.0,3.0,0.0,0.0,0.0,0.0,0.0,63.0,0.0,0.0,0.0,250.420365,0.0
4,0.000060,219,219,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,168.0,3.0,0.0,0.0,0.0,0.0,0.0,62.0,0.0,0.0,0.0,-516.742119,0.0


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3, 
    random_state=42
)